# 7 · Evaluation & Comparison Plots

Aggregates the evaluated results into summary statistics and the comparison plots saved in `results/`.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from config import config

summary = {}
per_query = {}
relevance = {}
for key in config.PROFILES:
    data = json.loads((config.EVALUATED_DIR / f'{key}_final.json').read_text())
    ndcgs = [d['ndcg'] for d in data]
    rels = [s for d in data for s in d['relevance_scores']]
    per_query[key] = ndcgs
    relevance[key] = np.mean(rels) if rels else 0
    summary[key] = {'avg_ndcg': np.mean(ndcgs), 'std_ndcg': np.std(ndcgs),
                    'avg_relevance': relevance[key],
                    'avg_latency': np.mean([d['latency'] for d in data])}
pd.DataFrame(summary).T.round(4)

In [ ]:
labels = list(config.PROFILES)

plt.figure(figsize=(8,5))
plt.bar(labels, [summary[k]['avg_ndcg'] for k in labels])
plt.ylabel('Average nDCG'); plt.title('Average nDCG per Configuration'); plt.xticks(rotation=20)
plt.tight_layout(); plt.savefig(config.RESULTS_DIR / 'plot_avg_ndcg.png', dpi=150); plt.show()

plt.figure(figsize=(8,5))
plt.boxplot([per_query[k] for k in labels], labels=labels)
plt.ylabel('nDCG'); plt.title('nDCG Distribution'); plt.xticks(rotation=20)
plt.tight_layout(); plt.savefig(config.RESULTS_DIR / 'plot_ndcg_boxplot.png', dpi=150); plt.show()

plt.figure(figsize=(8,5))
plt.bar(labels, [relevance[k] for k in labels])
plt.ylabel('Average Relevance (0-5)'); plt.title('Average Relevance per Configuration'); plt.xticks(rotation=20)
plt.tight_layout(); plt.savefig(config.RESULTS_DIR / 'plot_avg_relevance.png', dpi=150); plt.show()